In [1]:
import sys
sys.path.append('../')

In [2]:
# Load models.
import torch
from genwarp import GenWarp

# ZoeDepth
zoedepth = torch.hub.load(
    '../extern/ZoeDepth',
    'ZoeD_N',
    source='local',
    pretrained=True,
    trust_repo=True
).to('cuda')

# GenWarp
genwarp_cfg = dict(
    pretrained_model_path='../checkpoints',
    checkpoint_name='multi1',
    half_precision_weights=True
)
genwarp_nvs = GenWarp(cfg=genwarp_cfg)

img_size [384, 512]


/home/genwarp/.local/lib/python3.10/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Params passed to Resize transform:
	width:  512
	height:  384
	resize_target:  True
	keep_aspect_ratio:  True
	ensure_multiple_of:  32
	resize_method:  minimal
Using pretrained resource url::https://github.com/isl-org/ZoeDepth/releases/download/v1.0/ZoeD_M12_N.pt
Loaded successfully
Loading GenWarp...


/home/genwarp/.local/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Loaded GenWarp.


In [3]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os, gc, re, csv, traceback
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
from torchvision.transforms.functional import to_tensor, to_pil_image
from glob import glob
import matplotlib.cm as mcm

from genwarp.ops import (
    camera_lookat, get_projection_matrix, sph2cart, focal_length_to_fov
)

# ── Settings ──────────────────────────────────────────────────────────────────
synthesized_dir = "../st_50_out_low"
source_dir      = "../st_50"
out_dir         = "../hagva_module_a/stlucia/signal1_geometric"
focal_length_mm = 19
near, far       = 0.01, 100
azi_deg         = 0.0

# ── Helper functions ──────────────────────────────────────────────────────────

def pad_to_multiple(tensor, multiple=64):
    """Pad BCHW tensor so H and W are multiples of `multiple`.
    Returns (padded_tensor, pad_h, pad_w).
    """
    B, C, H, W = tensor.shape
    pad_h = (multiple - H % multiple) % multiple
    pad_w = (multiple - W % multiple) % multiple
    if pad_h > 0 or pad_w > 0:
        tensor = F.pad(tensor, (0, pad_w, 0, pad_h))
    return tensor, pad_h, pad_w


def unpad_pil(pil_img, orig_h, orig_w):
    """Crop a PIL image back to (orig_h, orig_w) after model padding."""
    return pil_img.crop((0, 0, orig_w, orig_h))


def cleanup_vars():
    """Flush CPU and GPU memory caches."""
    gc.collect()
    torch.cuda.empty_cache()


def parse_synth_filename(filename):
    """
    Parse filenames like '01_Image001_ele1_r0.10.jpg'.
    Returns (ele_deg, radius, image_stem) or None on failure.
    image_stem is the source stem with leading index prefix stripped.
    """
    stem = os.path.splitext(filename)[0]
    match = re.search(r'_ele([\.\d]+)_r([\d.]+)$', stem)
    if match is None:
        return None
    ele_deg    = float(match.group(1))
    radius     = float(match.group(2))
    base       = stem[:match.start()]
    image_stem = re.sub(r'^\d+_', '', base)
    return ele_deg, radius, image_stem


def compute_geometric_score(synthesized_np, warped_np, mask_np):
    """
    Per-pixel geometric consistency score in [0, 1].

    Args:
        synthesized_np : (H, W, 3) float32 in [0, 1]  — synthesized image
        warped_np      : (H, W, 3) float32 in [0, 1]  — GenWarp warp
        mask_np        : (H, W)    float32 in [0, 1]  — 1 = covered by warp

    Returns:
        score_map : (H, W) float32 in [0, 1]
                    0 = consistent, 1 = hallucinated / uncovered
    """
    H, W = mask_np.shape
    score_map = np.ones((H, W), dtype=np.float32)  # default: worst score

    visible = mask_np >= 0.5
    if visible.any():
        diff         = np.abs(synthesized_np - warped_np).mean(axis=2)  # (H,W)
        visible_diff = diff[visible]
        p99 = np.percentile(visible_diff, 99)
        if p99 > 0:
            diff_norm = np.clip(diff / p99, 0.0, 1.0)
        else:
            diff_norm = np.zeros_like(diff)
        score_map[visible] = diff_norm[visible]
        # pixels with mask < 0.5 keep score = 1.0 (hallucinated / no coverage)

    return score_map


def score_to_colormap(score_map):
    """Convert [0,1] score map to RGB uint8 using RdYlBu_r (blue=safe, red=hallucinated)."""
    colored = mcm.RdYlBu_r(score_map)          # (H, W, 4) float64
    return (colored[..., :3] * 255).astype(np.uint8)


# ── Camera setup constants ────────────────────────────────────────────────────
device = 'cuda'
azi    = torch.tensor(np.deg2rad(azi_deg))
z_up   = torch.tensor([[0., 0., 1.]])

# Reference view matrix — world: z-up, y-right, x-back (ZoeDepth convention)
src_view_mtx = camera_lookat(
    torch.tensor([[0., 0., 0.]]),   # eye at origin
    torch.tensor([[-1., 0., 0.]]),  # looking toward -x
    z_up
)

fovy = focal_length_to_fov(focal_length_mm, 24.)
fovy = torch.ones(1) * fovy

# ── Build source image lookup: stem -> full path ──────────────────────────────
source_lookup = {}
for _p in glob(os.path.join(source_dir, '*')):
    if os.path.isfile(_p):
        _stem = os.path.splitext(os.path.basename(_p))[0]
        source_lookup[_stem] = _p

print(f"Found {len(source_lookup)} source images in {source_dir}")

# ── Initialise CSV log ────────────────────────────────────────────────────────
os.makedirs(out_dir, exist_ok=True)
csv_path = os.path.join(out_dir, 'signal1_summary.csv')
with open(csv_path, 'w', newline='') as _f:
    csv.writer(_f).writerow(
        ['synth_file', 'source_file', 'ele_deg', 'radius',
         'mean_geo_score', 'pct_hallucinated']
    )

# ── Main loop ─────────────────────────────────────────────────────────────────
synth_files = sorted(
    glob(os.path.join(synthesized_dir, '*.jpg')) +
    glob(os.path.join(synthesized_dir, '*.png'))
)

processed, failed = 0, 0

for idx, synth_path in enumerate(synth_files):
    synth_fname = os.path.basename(synth_path)

    # Skip r15.00 images
    if 'r15.00' in synth_fname:
        continue

    warped_t = mask_t = src_tensor = src_depth = renders = None
    rel_view_mtx = tar_view_mtx = src_proj_mtx = tar_proj_mtx = None

    try:
        # Parse filename
        parsed = parse_synth_filename(synth_fname)
        if parsed is None:
            print(f"[{idx}] SKIP (parse failed): {synth_fname}")
            continue
        ele_deg, radius, image_stem = parsed

        # Find matching source image
        if image_stem not in source_lookup:
            print(f"[{idx}] SKIP (source not found '{image_stem}'): {synth_fname}")
            continue
        src_path = source_lookup[image_stem]

        # Load source image
        src_pil        = Image.open(src_path).convert('RGB')
        orig_w, orig_h = src_pil.size
        src_tensor     = to_tensor(src_pil)[None].to(device)  # 1CHW float32

        # Depth estimation
        with torch.no_grad():
            src_depth = zoedepth.infer(src_tensor)  # 1CHW float32

        # Pad to multiple of 64 and go half-precision
        src_tensor, pad_h, pad_w = pad_to_multiple(src_tensor.half())
        src_depth,  _,     _     = pad_to_multiple(src_depth.half())

        _, _, pH, pW = src_tensor.shape
        aspect_wh    = pW / pH

        # Projection matrices
        src_proj_mtx = get_projection_matrix(
            fovy=fovy, aspect_wh=aspect_wh, near=near, far=far
        ).to(src_tensor)
        tar_proj_mtx = src_proj_mtx

        # Target view matrix
        ele          = torch.tensor(np.deg2rad(ele_deg))
        mean_depth   = src_depth.mean(dim=(2, 3)).squeeze(1).cpu()
        eye          = sph2cart(azi, ele, mean_depth + radius).float()
        at           = F.pad(-mean_depth[:, None], (0, 2), mode='constant', value=0)
        tar_view_mtx = camera_lookat(eye + at, at, z_up)
        rel_view_mtx = (
            tar_view_mtx @ torch.linalg.inv(src_view_mtx.float())
        ).to(src_tensor)

        # Run GenWarp — extract only warped and mask
        with torch.no_grad():
            renders = genwarp_nvs(
                src_image    = src_tensor,
                src_depth    = src_depth,
                rel_view_mtx = rel_view_mtx,
                src_proj_mtx = src_proj_mtx,
                tar_proj_mtx = tar_proj_mtx
            )
        warped_t = renders['warped']   # 1CHW float16
        mask_t   = renders['mask']     # 1(C)HW float16

        # Free GenWarp inputs — no longer needed
        del renders, src_tensor, src_depth
        del rel_view_mtx, tar_view_mtx, src_proj_mtx, tar_proj_mtx
        renders = src_tensor = src_depth = None
        rel_view_mtx = tar_view_mtx = src_proj_mtx = tar_proj_mtx = None
        gc.collect()
        torch.cuda.empty_cache()

        # Load existing synthesized image
        synth_pil              = Image.open(synth_path).convert('RGB')
        synth_orig_w, synth_orig_h = synth_pil.size

        # Convert warped tensor -> PIL, unpad, resize to synthesized dimensions
        warped_pil = unpad_pil(
            to_pil_image(warped_t[0].float().clamp(0, 1)),
            orig_h, orig_w
        )
        warped_pil = warped_pil.resize((synth_orig_w, synth_orig_h), Image.BILINEAR)

        # Convert mask tensor -> PIL, unpad, resize
        mask_arr = mask_t[0].float().cpu().numpy()
        if mask_arr.ndim == 3:
            mask_arr = mask_arr[0]  # CHW -> HW
        mask_pil = Image.fromarray(
            np.clip(mask_arr * 255, 0, 255).astype(np.uint8)
        )
        mask_pil = unpad_pil(mask_pil, orig_h, orig_w)
        mask_pil = mask_pil.resize((synth_orig_w, synth_orig_h), Image.NEAREST)

        # Free GPU tensors — fully converted to PIL/numpy now
        del warped_t, mask_t
        warped_t = mask_t = None
        gc.collect()
        torch.cuda.empty_cache()

        # Build numpy arrays for scoring
        synth_np  = np.array(synth_pil,  dtype=np.float32) / 255.
        warped_np = np.array(warped_pil, dtype=np.float32) / 255.
        mask_np   = np.array(mask_pil,   dtype=np.float32) / 255.

        # Compute geometric score
        score_map      = compute_geometric_score(synth_np, warped_np, mask_np)
        mean_geo_score = float(score_map.mean())
        pct_hall       = float((score_map > 0.5).mean() * 100.)

        # Save outputs
        stem_out = image_stem
        warped_pil.save(os.path.join(out_dir, f"{stem_out}_warped.jpg"))
        mask_pil.save(  os.path.join(out_dir, f"{stem_out}_mask.png"))
        np.save(        os.path.join(out_dir, f"{stem_out}_geo_score.npy"), score_map)
        Image.fromarray(score_to_colormap(score_map)).save(
            os.path.join(out_dir, f"{stem_out}_geo_score_viz.jpg")
        )

        # Append CSV row
        with open(csv_path, 'a', newline='') as _f:
            csv.writer(_f).writerow([
                synth_fname, os.path.basename(src_path),
                ele_deg, radius,
                f"{mean_geo_score:.4f}", f"{pct_hall:.2f}"
            ])

        print(f"[{idx}] {synth_fname} | ele={ele_deg} r={radius} "
              f"| geo={mean_geo_score:.4f} hall={pct_hall:.2f}%")
        processed += 1

    except Exception:
        print(f"[{idx}] ERROR: {synth_fname}")
        traceback.print_exc()
        failed += 1

    finally:
        del src_tensor, src_depth, renders, warped_t, mask_t
        del rel_view_mtx, tar_view_mtx, src_proj_mtx, tar_proj_mtx
        cleanup_vars()

print(f"\nDone. processed={processed}, failed={failed}")
print(f"Outputs -> {out_dir}")


Found 50 source images in ../st_50


It looks like you are trying to rescale already rescaled images. If the input images have pixel values between 0 and 1, set `do_rescale=False` to avoid rescaling them again.


[0] 01_image018451_ele1_r0.10.jpg | ele=1.0 r=0.1 | geo=0.3932 hall=34.71%
[1] 02_image018526_ele3_r0.00.jpg | ele=3.0 r=0.0 | geo=0.3621 hall=31.52%
[2] 03_image018541_ele2_r0.00.jpg | ele=2.0 r=0.0 | geo=0.3388 hall=27.21%
[3] 04_image018556_ele1_r0.10.jpg | ele=1.0 r=0.1 | geo=0.3986 hall=31.30%
[4] 05_image018601_ele5_r0.20.jpg | ele=5.0 r=0.2 | geo=0.4082 hall=36.45%
[5] 06_image018766_ele1_r0.10.jpg | ele=1.0 r=0.1 | geo=0.3962 hall=30.51%
[6] 07_image018781_ele1_r0.00.jpg | ele=1.0 r=0.0 | geo=0.3803 hall=31.26%
[7] 08_image018796_ele2_r0.10.jpg | ele=2.0 r=0.1 | geo=0.4163 hall=34.42%
[8] 09_image018826_ele5_r0.00.jpg | ele=5.0 r=0.0 | geo=0.4065 hall=34.24%
[9] 10_image018856_ele5_r0.20.jpg | ele=5.0 r=0.2 | geo=0.4269 hall=36.29%
[10] 11_image018961_ele2_r0.20.jpg | ele=2.0 r=0.2 | geo=0.4175 hall=35.85%
[12] 13_image019036_ele1_r0.00.jpg | ele=1.0 r=0.0 | geo=0.4084 hall=33.04%
[14] 15_image019261_ele3_r0.00.jpg | ele=3.0 r=0.0 | geo=0.3960 hall=32.06%
[16] 17_image019291_el

In [4]:
# ── Additional imports ───────────────────────────────────────────────────────
import torchvision.transforms as T

# ── Settings ──────────────────────────────────────────────────────────────────
synthesized_dir = "../st_50_out_low"
source_dir      = "../st_50"
signal1_dir     = "../hagva_module_a/stlucia/signal1_geometric"
out_dir         = "../hagva_module_a/stlucia/signal2_semantic"

# ── Load DINOv2 ───────────────────────────────────────────────────────────────
dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device).eval()

# DINOv2 preprocessing: 448x448 = 32x32 patches at patch_size=14
dino_transform = T.Compose([
    T.Resize((448, 448)),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

# ── Helper functions ──────────────────────────────────────────────────────────

def extract_dino_features(pil_img, dino_model, transform, device):
    """
    Extract DINOv2 patch features from a PIL image.

    Returns:
        feat_map : (H_patches, W_patches, feat_dim) float32 on CPU
                   For 448x448 input with patch_size=14: (32, 32, 384)
    """
    x = transform(pil_img).unsqueeze(0).to(device)   # 1CHW
    with torch.no_grad():
        # returns list of length n; each element is (B, num_patches, feat_dim)
        feats = dino_model.get_intermediate_layers(x, n=1, return_class_token=False)[0]
    # feats: (1, num_patches, feat_dim)
    feat_dim  = feats.shape[-1]
    num_patches = feats.shape[1]               # 32*32 = 1024
    h = w = int(num_patches ** 0.5)            # 32
    feat_map = feats[0].reshape(h, w, feat_dim).cpu().float()
    return feat_map


def compute_semantic_score(src_pil, synth_pil, dino_model, transform, device,
                            orig_h, orig_w):
    """
    Compute per-pixel semantic drift score in [0, 1].

    Extracts DINOv2 patch features from source and synthesized images,
    computes cosine distance per patch, upsamples to (orig_h, orig_w),
    and normalises by the 99th percentile.

    Returns:
        score_map : (orig_h, orig_w) float32  — high = drifted/hallucinated
    """
    src_feat   = extract_dino_features(src_pil,   dino_model, transform, device)
    synth_feat = extract_dino_features(synth_pil, dino_model, transform, device)

    # Cosine similarity per patch: (H_p, W_p)
    cos_sim = F.cosine_similarity(
        src_feat.unsqueeze(0),    # (1, H_p, W_p, D)
        synth_feat.unsqueeze(0),  # (1, H_p, W_p, D)
        dim=-1
    ).squeeze(0)                  # (H_p, W_p)

    cos_dist = (1.0 - cos_sim).clamp(0.0, 2.0)   # [0, 2]; 0 = identical

    # Upsample to original image resolution via bilinear interpolation
    dist_up = F.interpolate(
        cos_dist.unsqueeze(0).unsqueeze(0),        # (1, 1, H_p, W_p)
        size=(orig_h, orig_w),
        mode='bilinear',
        align_corners=False
    ).squeeze().numpy()                             # (orig_h, orig_w)

    # Normalise to [0, 1] using 99th percentile
    p99 = float(np.percentile(dist_up, 99))
    if p99 > 0:
        score_map = np.clip(dist_up / p99, 0.0, 1.0).astype(np.float32)
    else:
        score_map = np.zeros_like(dist_up, dtype=np.float32)

    return score_map


def score_to_colormap(score_map):
    """Convert [0,1] score map to RGB uint8 using RdYlBu_r (blue=safe, red=drifted)."""
    import matplotlib.cm as mcm
    colored = mcm.RdYlBu_r(score_map)
    return (colored[..., :3] * 255).astype(np.uint8)


# ── Initialise CSV log ────────────────────────────────────────────────────────
os.makedirs(out_dir, exist_ok=True)
csv_path = os.path.join(out_dir, 'signal2_summary.csv')
with open(csv_path, 'w', newline='') as _f:
    csv.writer(_f).writerow(
        ['synth_file', 'source_file', 'ele_deg', 'radius', 'mean_semantic_score']
    )

# ── Main loop ─────────────────────────────────────────────────────────────────
synth_files = sorted(
    glob(os.path.join(synthesized_dir, '*.jpg')) +
    glob(os.path.join(synthesized_dir, '*.png'))
)

processed, failed = 0, 0

for idx, synth_path in enumerate(synth_files):
    synth_fname = os.path.basename(synth_path)

    # Skip r15.00 images
    if 'r15.00' in synth_fname:
        continue

    try:
        # Parse filename (reuses Signal 1 helper)
        parsed = parse_synth_filename(synth_fname)
        if parsed is None:
            print(f"[{idx}] SKIP (parse failed): {synth_fname}")
            continue
        ele_deg, radius, image_stem = parsed

        # Find matching source image
        if image_stem not in source_lookup:
            print(f"[{idx}] SKIP (source not found '{image_stem}'): {synth_fname}")
            continue
        src_path = source_lookup[image_stem]

        # Load images as PIL
        src_pil   = Image.open(src_path).convert('RGB')
        synth_pil = Image.open(synth_path).convert('RGB')
        orig_w, orig_h = src_pil.size

        # Compute semantic score
        score_map        = compute_semantic_score(
            src_pil, synth_pil, dino, dino_transform, device, orig_h, orig_w
        )
        mean_sem_score   = float(score_map.mean())

        # Load Signal 1 geo score (if present — for future fusion; not used here)
        geo_score_path = os.path.join(signal1_dir, f"{image_stem}_geo_score.npy")
        if os.path.exists(geo_score_path):
            _geo_score = np.load(geo_score_path)   # available for downstream use
        else:
            _geo_score = None

        # Save outputs
        np.save(os.path.join(out_dir, f"{image_stem}_sem_score.npy"), score_map)
        Image.fromarray(score_to_colormap(score_map)).save(
            os.path.join(out_dir, f"{image_stem}_sem_score_viz.jpg")
        )

        # Append CSV row
        with open(csv_path, 'a', newline='') as _f:
            csv.writer(_f).writerow([
                synth_fname, os.path.basename(src_path),
                ele_deg, radius,
                f"{mean_sem_score:.4f}"
            ])

        print(f"[{idx}] {synth_fname} | ele={ele_deg} r={radius} "
              f"| sem={mean_sem_score:.4f}")
        processed += 1

    except Exception:
        print(f"[{idx}] ERROR: {synth_fname}")
        traceback.print_exc()
        failed += 1

    finally:
        gc.collect()
        torch.cuda.empty_cache()

print(f"\nDone. processed={processed}, failed={failed}")
print(f"Outputs -> {out_dir}")


Using cache found in /home/genwarp/.cache/torch/hub/facebookresearch_dinov2_main
/home/genwarp/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/genwarp/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/genwarp/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


[0] 01_image018451_ele1_r0.10.jpg | ele=1.0 r=0.1 | sem=0.3751
[1] 02_image018526_ele3_r0.00.jpg | ele=3.0 r=0.0 | sem=0.4240
[2] 03_image018541_ele2_r0.00.jpg | ele=2.0 r=0.0 | sem=0.3954
[3] 04_image018556_ele1_r0.10.jpg | ele=1.0 r=0.1 | sem=0.3453
[4] 05_image018601_ele5_r0.20.jpg | ele=5.0 r=0.2 | sem=0.3332
[5] 06_image018766_ele1_r0.10.jpg | ele=1.0 r=0.1 | sem=0.3666
[6] 07_image018781_ele1_r0.00.jpg | ele=1.0 r=0.0 | sem=0.3460
[7] 08_image018796_ele2_r0.10.jpg | ele=2.0 r=0.1 | sem=0.4101
[8] 09_image018826_ele5_r0.00.jpg | ele=5.0 r=0.0 | sem=0.4020
[9] 10_image018856_ele5_r0.20.jpg | ele=5.0 r=0.2 | sem=0.3940
[10] 11_image018961_ele2_r0.20.jpg | ele=2.0 r=0.2 | sem=0.3885
[12] 13_image019036_ele1_r0.00.jpg | ele=1.0 r=0.0 | sem=0.2933
[14] 15_image019261_ele3_r0.00.jpg | ele=3.0 r=0.0 | sem=0.3595
[16] 17_image019291_ele1_r0.10.jpg | ele=1.0 r=0.1 | sem=0.3670
[17] 18_image019321_ele4_r0.10.jpg | ele=4.0 r=0.1 | sem=0.3856
[20] 21_image019486_ele1_r0.20.jpg | ele=1.0 r=0.2

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# HAGVA Module A — Validation
# Task 1: Side-by-side visual comparison for red regions (top-5 by geo score)
# Task 2: Combined Signal 1 + Signal 2 map for all 36 images
# ══════════════════════════════════════════════════════════════════════════════
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as mcm
from PIL import Image

signal1_dir = "../hagva_module_a/stlucia/signal1_geometric"
signal2_dir = "../hagva_module_a/stlucia/signal2_semantic"
synth_dir   = "../st_50_out_low"
src_dir_val = "../st_50"
val_dir     = "../hagva_module_a/stlucia/validation"
device        = "cuda"
os.makedirs(val_dir, exist_ok=True)

# ── Pick top-5 images by mean_geo_score from signal1_summary.csv ──────────────
sig1_csv = os.path.join(signal1_dir, 'signal1_summary.csv')
with open(sig1_csv, newline='') as _f:
    rows = list(csv.DictReader(_f))

rows_valid = [r for r in rows if r['mean_geo_score']]
rows_valid.sort(key=lambda r: float(r['mean_geo_score']), reverse=True)
top5 = rows_valid[:5]

print("Top-5 images by geometric score:")
for r in top5:
    print(f"  {r['synth_file']}  geo={r['mean_geo_score']}")

# ── Task 1: 3-panel comparison figures ───────────────────────────────────────
print("\n── Task 1: side-by-side comparison figures ──")

for row in top5:
    synth_fname = row['synth_file']
    parsed = parse_synth_filename(synth_fname)
    if parsed is None:
        print(f"  SKIP (parse failed): {synth_fname}")
        continue
    _, _, image_stem = parsed

    try:
        src_path      = source_lookup.get(image_stem)
        synth_path    = os.path.join(synth_dir, synth_fname)
        geo_viz_path  = os.path.join(signal1_dir, f"{image_stem}_geo_score_viz.jpg")

        if src_path is None or not os.path.exists(synth_path) or not os.path.exists(geo_viz_path):
            print(f"  SKIP (missing files): {image_stem}")
            continue

        src_img     = Image.open(src_path).convert('RGB')
        synth_img   = Image.open(synth_path).convert('RGB')
        geo_viz_img = Image.open(geo_viz_path).convert('RGB')

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(src_img);     axes[0].set_title("Source",                        fontsize=13)
        axes[1].imshow(synth_img);   axes[1].set_title("Synthesized (GenWarp)",         fontsize=13)
        axes[2].imshow(geo_viz_img); axes[2].set_title("Geometric Score (red=hallucinated)", fontsize=13)
        for ax in axes:
            ax.axis('off')
        fig.suptitle(image_stem, fontsize=11, y=0.02)
        fig.tight_layout()

        out_path = os.path.join(val_dir, f"{image_stem}_comparison.jpg")
        fig.savefig(out_path, bbox_inches='tight', dpi=120)
        plt.close(fig)
        print(f"  Saved: {os.path.basename(out_path)}")

    except Exception:
        print(f"  ERROR: {image_stem}")
        traceback.print_exc()

# ── Task 2: Combined score map for all 36 images ──────────────────────────────
print("\n── Task 2: combined Signal 1 + Signal 2 maps ──")

combined_csv_path = os.path.join(val_dir, 'combined_summary.csv')
with open(combined_csv_path, 'w', newline='') as _f:
    csv.writer(_f).writerow(
        ['image_stem', 'mean_geo_score', 'mean_sem_score', 'mean_combined_score']
    )

# Collect all stems that have both signal files
geo_files = glob(os.path.join(signal1_dir, '*_geo_score.npy'))
stems_all = sorted(
    os.path.basename(p).replace('_geo_score.npy', '') for p in geo_files
)

proc2, fail2 = 0, 0

# Track which stems are in the top-5 for the 5-image combined figure
top5_stems = set()
for row in top5:
    parsed = parse_synth_filename(row['synth_file'])
    if parsed:
        top5_stems.add(parsed[2])

for image_stem in stems_all:
    geo_path = os.path.join(signal1_dir, f"{image_stem}_geo_score.npy")
    sem_path = os.path.join(signal2_dir, f"{image_stem}_sem_score.npy")
    geo_viz  = os.path.join(signal1_dir, f"{image_stem}_geo_score_viz.jpg")
    sem_viz  = os.path.join(signal2_dir, f"{image_stem}_sem_score_viz.jpg")

    if not os.path.exists(sem_path):
        print(f"  SKIP (no sem score): {image_stem}")
        continue

    try:
        geo_score = np.load(geo_path).astype(np.float32)
        sem_score = np.load(sem_path).astype(np.float32)

        # Resize sem to geo shape if they differ
        if geo_score.shape != sem_score.shape:
            sem_pil   = Image.fromarray((sem_score * 255).clip(0, 255).astype(np.uint8))
            sem_pil   = sem_pil.resize((geo_score.shape[1], geo_score.shape[0]), Image.BILINEAR)
            sem_score = np.array(sem_pil, dtype=np.float32) / 255.

        combined = ((geo_score + sem_score) / 2.).astype(np.float32)

        mean_geo  = float(geo_score.mean())
        mean_sem  = float(sem_score.mean())
        mean_comb = float(combined.mean())

        # Save combined score and viz
        np.save(os.path.join(val_dir, f"{image_stem}_combined_score.npy"), combined)
        Image.fromarray(score_to_colormap(combined)).save(
            os.path.join(val_dir, f"{image_stem}_combined_viz.jpg")
        )

        # 3-panel: geo viz | sem viz | combined viz
        if os.path.exists(geo_viz) and os.path.exists(sem_viz):
            geo_img  = Image.open(geo_viz).convert('RGB')
            sem_img  = Image.open(sem_viz).convert('RGB')
            comb_img = Image.fromarray(score_to_colormap(combined))

            fig, axes = plt.subplots(1, 3, figsize=(15, 5))
            axes[0].imshow(geo_img);  axes[0].set_title("Signal 1 — Geometric",  fontsize=13)
            axes[1].imshow(sem_img);  axes[1].set_title("Signal 2 — Semantic",   fontsize=13)
            axes[2].imshow(comb_img); axes[2].set_title("Combined (avg)",         fontsize=13)
            for ax in axes:
                ax.axis('off')
            fig.suptitle(image_stem, fontsize=11, y=0.02)
            fig.tight_layout()
            fig.savefig(
                os.path.join(val_dir, f"{image_stem}_signals_combined.jpg"),
                bbox_inches='tight', dpi=120
            )
            plt.close(fig)

        # CSV
        with open(combined_csv_path, 'a', newline='') as _f:
            csv.writer(_f).writerow([
                image_stem,
                f"{mean_geo:.4f}", f"{mean_sem:.4f}", f"{mean_comb:.4f}"
            ])

        tag = " ← top-5" if image_stem in top5_stems else ""
        print(f"  {image_stem}  geo={mean_geo:.4f}  sem={mean_sem:.4f}  comb={mean_comb:.4f}{tag}")
        proc2 += 1

    except Exception:
        print(f"  ERROR: {image_stem}")
        traceback.print_exc()
        fail2 += 1

    finally:
        gc.collect()

print(f"\nTask 2 done. processed={proc2}, failed={fail2}")
print(f"Validation outputs -> {val_dir}")
print(f"Combined summary   -> {combined_csv_path}")


Top-5 images by geometric score:
  42_image021031_ele3_r0.20.jpg  geo=0.4771
  41_image020956_ele2_r0.10.jpg  geo=0.4699
  21_image019486_ele1_r0.20.jpg  geo=0.4610
  17_image019291_ele1_r0.10.jpg  geo=0.4609
  30_image020146_ele1_r0.20.jpg  geo=0.4584

── Task 1: side-by-side comparison figures ──
  Saved: image021031_comparison.jpg
  Saved: image020956_comparison.jpg
  Saved: image019486_comparison.jpg
  Saved: image019291_comparison.jpg
  Saved: image020146_comparison.jpg

── Task 2: combined Signal 1 + Signal 2 maps ──
  image018451  geo=0.3932  sem=0.3751  comb=0.3841
  image018526  geo=0.3621  sem=0.4240  comb=0.3931
  image018541  geo=0.3388  sem=0.3954  comb=0.3671
  image018556  geo=0.3986  sem=0.3453  comb=0.3719
  image018601  geo=0.4082  sem=0.3332  comb=0.3707
  image018766  geo=0.3962  sem=0.3666  comb=0.3814
  image018781  geo=0.3803  sem=0.3460  comb=0.3632
  image018796  geo=0.4163  sem=0.4101  comb=0.4132
  image018826  geo=0.4065  sem=0.4020  comb=0.4042
  image01885